In [1]:
%load_ext autoreload

In [2]:
%autoreload 2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%aimport plot_AUPRC

In [5]:
# 1. Loading in predictor outputs on Lambert TFs
yeast_TFs = pd.read_csv("../output/yeast_TF_seqs.csv", index_col = 0)
yeast_TFs = yeast_TFs.rename(columns = {"seq" : "sequence"})
yeast_TFs["uniprotID"] = yeast_TFs["id"].str.split("|").str[1]
yeast_TFs["length"] = yeast_TFs["sequence"].str.len()
yeast_TFs

,id,sequence,uniprotID,length
0,sp|O14467|MBF1_YEAST Multiprotein-bridging fac...,MSDWDTNTIIGSRARAGGSGPRANVARSQGQINAARRQGLVVSVDK...,O14467,151
1,sp|O93958|MATA2_YARLL Mating-type protein A2 O...,MENTILHIHSFQLPQTEQPYPEAMLFDRDTSDSRTVLTQKPNGLEI...,O93958,291
2,sp|P03069|GCN4_YEAST General control transcrip...,MSEYQPSLFALNPMGFSPLDGSKSTNENVSASTSTAKPMVGQLIFD...,P03069,281
3,sp|P04386|GAL4_YEAST Regulatory protein GAL4 O...,MKLLSSIEQACDICRLKKLKCSKEKPKCAKCLKNNWECRYSPKTKR...,P04386,881
4,sp|P04387|GAL80_YEAST Galactose/lactose metabo...,MDYNKRSSVSTVPNAAPIRVGFVGLNAAKGWAIKTHYPAILQLSSQ...,P04387,435
...,...,...,...,...
242,sp|Q707Y3|MATA1_YARLL Mating-type protein A1 O...,MPSRTPTDIWRCQRLILAARKGETTCQALHEQSIEISSSLKWFEEI...,Q707Y3,176
243,sp|Q707Y6|MATA1_PICAN Mating-type protein A1 O...,MQFTILNEPSLDSQRREGDLASENYVFGDIRKEGVRILEDSLRSER...,Q707Y6,181
244,sp|Q708A1|MATA1_NAKDE Mating-type protein A1 O...,MNVQEIHNIREACITILSGTKHNSVLFEPCDKFDEVINSLDIDPDS...,Q708A1,122
245,sp|Q9HG12|MATA1_KLULA Mating-type protein A1 O...,MCDNDMADIQSKLSSFCEEIRALALKEGYNLEGDKSPSSKPYFMSW...,Q9HG12,228


In [6]:
known_ADs = pd.read_excel("../data/sanborn_known_ADs_elife-68068-fig1-data2-v3.xlsx")
known_ADs  = known_ADs.rename(columns = {"protein ID": "uniprotID", "start" : "Start", "stop" : "End"})
known_ADs

,protein,uniprotID,Start,End,maximal activation,maximal Z score,maximal fragment,sequence
0,GCN4,P03069,63,142,128.273405,11.910003,A_tiles_P03069:76,NLDFDFALPQTATAPDAKTVLPIPELDDAVVESFFSSSTDSTPMFE...
1,GAL4,P04386,832,880,44.019965,9.285866,A_tiles_P04386:828,SKPLSPGWTDQTAYNAFGITTGMFNTTTMDDVYNYLFDDEDTPPNPKK
2,GAL4,P04386,146,199,59.660889,10.031831,A_controls-mean_GAL4,SIDSAAHHDNSTIPLDFMPRDALHGFDWSEEDDMSDGLPFLKTDPN...
3,ARG81,P05085,194,256,32.070085,8.508774,A_tiles_P05085:194,KGHVKTGILSANDGVPPTPNLLDYDWNNLNITGYEWISSELRDDAL...
4,ARG81,P05085,69,126,13.391200,6.366012,A_tiles_P05085:65,IPQNSPATTTNLSGSVDEPQYQRRNIDFVRYDEEYVYHEDMDDELT...
...,...,...,...,...,...,...,...,...
145,YRM1,Q12340,737,785,73.486889,10.543228,A_tiles_Q12340:733,SENASHNNETGPIETELAQTISNEFWTAYNLGWEELMSQPDYKYLFDT
146,YRM1,Q12340,184,241,21.275056,7.501859,A_tiles_Q12340:193,PLEKTGSDILQQVCNVLPSFEQSSKIITDFFNTELETNEVSEVLDK...
147,HAA1,Q12753,658,693,35.587427,8.764113,A_tiles_Q12753:641,DLPDTSPMSSIQTASPPSQLLTDQGFADLDNFMSS
148,HAA1,Q12753,274,335,28.366703,8.207702,A_tiles_Q12753:282,NSRVGEVSVPLEEYIPSDIDGVGRVTDKSSLVYDWPFDESIERNFS...


In [8]:
# Only keep ADs on yeast TFs
all_known_ADs = pd.read_csv("../../SFARI/output/known_ADs_considering_isoforms_and_canonical_with_alerasool.csv")
known_ADs = known_ADs[known_ADs["uniprotID"].isin(yeast_TFs["uniprotID"])]
known_ADs

,protein,uniprotID,Start,End,maximal activation,maximal Z score,maximal fragment,sequence
0,GCN4,P03069,63,142,128.273405,11.910003,A_tiles_P03069:76,NLDFDFALPQTATAPDAKTVLPIPELDDAVVESFFSSSTDSTPMFE...
1,GAL4,P04386,832,880,44.019965,9.285866,A_tiles_P04386:828,SKPLSPGWTDQTAYNAFGITTGMFNTTTMDDVYNYLFDDEDTPPNPKK
2,GAL4,P04386,146,199,59.660889,10.031831,A_controls-mean_GAL4,SIDSAAHHDNSTIPLDFMPRDALHGFDWSEEDDMSDGLPFLKTDPN...
3,ARG81,P05085,194,256,32.070085,8.508774,A_tiles_P05085:194,KGHVKTGILSANDGVPPTPNLLDYDWNNLNITGYEWISSELRDDAL...
4,ARG81,P05085,69,126,13.391200,6.366012,A_tiles_P05085:65,IPQNSPATTTNLSGSVDEPQYQRRNIDFVRYDEEYVYHEDMDDELT...
...,...,...,...,...,...,...,...,...
145,YRM1,Q12340,737,785,73.486889,10.543228,A_tiles_Q12340:733,SENASHNNETGPIETELAQTISNEFWTAYNLGWEELMSQPDYKYLFDT
146,YRM1,Q12340,184,241,21.275056,7.501859,A_tiles_Q12340:193,PLEKTGSDILQQVCNVLPSFEQSSKIITDFFNTELETNEVSEVLDK...
147,HAA1,Q12753,658,693,35.587427,8.764113,A_tiles_Q12753:641,DLPDTSPMSSIQTASPPSQLLTDQGFADLDNFMSS
148,HAA1,Q12753,274,335,28.366703,8.207702,A_tiles_Q12753:282,NSRVGEVSVPLEEYIPSDIDGVGRVTDKSSLVYDWPFDESIERNFS...


In [10]:
# Preparing known AD coords for merge
known_AD_coords = known_ADs[["uniprotID", "Start", "End"]]
known_AD_coords = known_AD_coords.rename(columns = {"Start" : "annot_Start", "End" : "annot_End"})
known_AD_coords


,uniprotID,annot_Start,annot_End
0,P03069,63,142
1,P04386,832,880
2,P04386,146,199
3,P05085,194,256
4,P05085,69,126
...,...,...,...
145,Q12340,737,785
146,Q12340,184,241
147,Q12753,658,693
148,Q12753,274,335


In [18]:
tada = pd.read_csv("../output/tada_pa_fix_yeast/TADA_preds.csv")
tada["predictor"] = "tada"
tada

,Unnamed: 0,sequence,tada_centers,tada_preds,predictor
0,0,MSDWDTNTIIGSRARAGGSGPRANVARSQGQINAARRQGLVVSVDK...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.1741496,0.17925483,0.17271048,0.16345137,0.1...",tada
1,1,MENTILHIHSFQLPQTEQPYPEAMLFDRDTSDSRTVLTQKPNGLEI...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.33071885,0.29227117,0.27643162,0.2779289,0.2...",tada
2,2,MSEYQPSLFALNPMGFSPLDGSKSTNENVSASTSTAKPMVGQLIFD...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.2735822,0.253328,0.24093209,0.23345102,0.225...",tada
3,3,MKLLSSIEQACDICRLKKLKCSKEKPKCAKCLKNNWECRYSPKTKR...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.15183689,0.15342252,0.14922355,0.14678252,0....",tada
4,4,MDYNKRSSVSTVPNAAPIRVGFVGLNAAKGWAIKTHYPAILQLSSQ...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.16549951,0.16396415,0.15702224,0.15454084,0....",tada
...,...,...,...,...,...
240,240,MPSRTPTDIWRCQRLILAARKGETTCQALHEQSIEISSSLKWFEEI...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.16377363,0.15304711,0.16052012,0.20011984,0....",tada
241,241,MQFTILNEPSLDSQRREGDLASENYVFGDIRKEGVRILEDSLRSER...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.23934273,0.26260874,0.29964358,0.26272327,0....",tada
242,242,MNVQEIHNIREACITILSGTKHNSVLFEPCDKFDEVINSLDIDPDS...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.28045863,0.31869313,0.38239524,0.43217668,0....",tada
243,243,MCDNDMADIQSKLSSFCEEIRALALKEGYNLEGDKSPSSKPYFMSW...,"20.0,21.0,22.0,23.0,24.0,25.0,26.0,27.0,28.0,2...","0.24723531,0.23033734,0.2299771,0.23082776,0.2...",tada


In [ ]:
pd.read_csv("../output/yeast_TFs_preds/")

TypeError: read_csv() missing 1 required positional argument: 'filepath_or_buffer'

In [20]:
! pwd

/Users/sanjanakotha/Desktop/Staller_Lab/consensus_predictor/notebooks


In [ ]:
# All yeast predictions
adhunter = pd.read_csv("../output/yeast_TFs_preds/cleaned/adhunter.csv", index_col=0)
adhunter["predictor"] = "adhunter"
adhunter
tada = pd.read_csv("../output/yeast_TFs_preds/cleaned/tada.csv", index_col=0)
tada["predictor"] = "tada"
tada
adpred = pd.read_csv("../output/yeast_TFs_preds/cleaned/adpred.csv", index_col=0)
adpred["predictor"] = "adpred"
adpred = adpred.rename(columns = {"GeneName" : "uniprotID"})
adpred
mechanistic = pd.read_csv("../output/yeast_TFs_preds/cleaned/composition.csv", index_col=0)
mechanistic["predictor"] = "mechanistic"
mechanistic
paddle = pd.read_csv("../output/yeast_TFs_preds/cleaned/paddle_noSS.csv", index_col=0)
paddle["predictor"] = "paddle"
paddle
all_models = pd.concat([adhunter, tada, adpred, mechanistic, paddle])
all_models
display(adhunter)
display(tada)
display(adpred)
display(mechanistic)
display(paddle)
display(all_models)

,uniprotID,Start,End,predictor
0,O93958,127,167,adhunter
1,O93958,186,291,adhunter
2,P03069,58,146,adhunter
3,P04386,134,210,adhunter
4,P04386,242,307,adhunter
...,...,...,...,...
447,Q12457,128,184,adhunter
448,Q12531,66,108,adhunter
449,Q707Y3,13,82,adhunter
450,Q707Y6,52,92,adhunter


,uniprotID,Start,End,predictor
0,O93958,192,290,tada
1,P03069,40,149,tada
2,P04386,135,207,tada
3,P04386,830,880,tada
4,P04387,364,410,tada
...,...,...,...,...
291,Q12363,49,94,tada
292,Q12457,129,180,tada
293,Q12753,272,337,tada
294,Q12753,651,694,tada


,uniprotID,Start,End,predictor
0,O93958,211,225,adpred
1,O93958,256,273,adpred
2,P03069,109,124,adpred
3,P04386,167,177,adpred
4,P04386,853,863,adpred
...,...,...,...,...
316,Q12340,360,374,adpred
317,Q12340,542,559,adpred
318,Q12340,752,780,adpred
319,Q707Y3,34,59,adpred


,uniprotID,Start,End,predictor
0,P33400,128,172,mechanistic
1,P33400,520,618,mechanistic
2,P13574,229,341,mechanistic
3,P53968,504,543,mechanistic
4,P21192,28,86,mechanistic
...,...,...,...,...
99,P38830,10,49,mechanistic
100,P14681,310,368,mechanistic
101,P32896,449,505,mechanistic
102,P26370,358,396,mechanistic


,uniprotID,Start,End,predictor
1,O93958,223,262,paddle
2,P03069,69,135,paddle
3,P04386,149,154,paddle
3,P04386,155,180,paddle
3,P04386,837,854,paddle
...,...,...,...,...
236,Q12363,317,339,paddle
236,Q12363,340,350,paddle
239,Q12457,143,175,paddle
241,Q12753,284,317,paddle


,uniprotID,Start,End,predictor
0,O93958,127,167,adhunter
1,O93958,186,291,adhunter
2,P03069,58,146,adhunter
3,P04386,134,210,adhunter
4,P04386,242,307,adhunter
...,...,...,...,...
236,Q12363,317,339,paddle
236,Q12363,340,350,paddle
239,Q12457,143,175,paddle
241,Q12753,284,317,paddle


In [ ]:
add_overlap_status(pred_output, known_AD_coords, "AD")